# Chess Dataset ML Analysis

Statistical analysis and machine learning on ~20,000 chess games.
Each section corresponds to a research question (RQ1–RQ7).

## Setup

Adds the project src/ to the path so we can import the chess_ml package.

In [ ]:
import os, sys
SRC = os.path.join(os.getcwd(), "..", "src")
sys.path.insert(0, SRC)

import pandas as pd
import numpy as np
from chess_ml.config import OUTPUT_FIGURES, OUTPUT_TABLES, MIN_GAMES_THRESHOLD
from chess_ml.data import load_data, preprocess
from chess_ml.analysis import (
    rq1_rating_diff_win_prob, rq2_top_openings, rq2_eco_analysis,
    rq3_duration_by_victory, rq4_rated_comparison,
    rq5_time_control, rq6_opening_ply,
)
from chess_ml.modeling import prepare_features, train_and_evaluate
from chess_ml.plotting import (
    plot_rating_diff, plot_openings, plot_duration,
    plot_rated_comparison, plot_win_distribution,
    plot_time_control, plot_opening_ply, plot_model_comparison,
    plot_confusion_matrices, plot_feature_importances, plot_eco_analysis,
)

os.makedirs(OUTPUT_FIGURES, exist_ok=True)
os.makedirs(OUTPUT_TABLES, exist_ok=True)

df_raw = load_data()
df = preprocess(df_raw)
print(f"Loaded {len(df_raw):,} rows, {len(df):,} after cleaning")

## RQ1: Rating Difference → Win Probability

Games are binned by (white_rating - black_rating). Error bars show 95% confidence intervals (Clopper-Pearson exact method).

In [ ]:
rq1 = rq1_rating_diff_win_prob(df)
rq1.to_csv(os.path.join(OUTPUT_TABLES, "RQ1_table.csv"), index=False)
display(rq1)
plot_rating_diff(rq1)

## RQ2: Opening Strategy → White Win Rate

Only openings with ≥30 games are considered. Green bars indicate the win rate is significantly above the overall mean (binomial test, p < 0.05).

In [ ]:
rq2 = rq2_top_openings(df)
rq2.to_csv(os.path.join(OUTPUT_TABLES, "RQ2_openings.csv"), index=False)
display(rq2)
plot_openings(rq2)

## RQ2b: ECO Opening Family Analysis

ECO codes are grouped into 5 families: Flank (A), Semi-Open (B), Open (C), Closed (D), Indian (E).

In [ ]:
rq2b = rq2_eco_analysis(df)
rq2b.to_csv(os.path.join(OUTPUT_TABLES, "RQ2_eco.csv"), index=False)
display(rq2b)
plot_eco_analysis(rq2b)

## RQ3: Game Duration by Victory Type

Error bars show ±1 standard error of the mean.

In [ ]:
rq3 = rq3_duration_by_victory(df)
rq3.to_csv(os.path.join(OUTPUT_TABLES, "RQ3_duration.csv"), index=False)
display(rq3)
plot_duration(rq3)

## RQ4: Rated vs Non-Rated Games

Welch's t-test and Mann-Whitney U test compare turn counts. Chi-square test compares win distributions.

In [ ]:
rq4_turns, rq4_dist, rq4_chi = rq4_rated_comparison(df)
rq4_turns.to_csv(os.path.join(OUTPUT_TABLES, "RQ4_rated.csv"), index=False)
rq4_dist.to_csv(os.path.join(OUTPUT_TABLES, "RQ4_win_distribution.csv"))
rq4_chi.to_csv(os.path.join(OUTPUT_TABLES, "RQ4_chi_square.csv"), index=False)
print("Turn comparison:")
display(rq4_turns)
print("
Win distribution:")
display(rq4_dist)
print("
Chi-square test:")
display(rq4_chi)
plot_rated_comparison(rq4_turns)
plot_win_distribution(rq4_dist)

## RQ5: Time Control → Game Length

Only time controls with ≥30 games are included.

In [ ]:
rq5 = rq5_time_control(df)
rq5.to_csv(os.path.join(OUTPUT_TABLES, "RQ5_time_control.csv"), index=False)
display(rq5)
plot_time_control(rq5)

## RQ6: Opening Depth → White Win Rate

Shaded region shows 95% confidence interval. Only depths with ≥30 games are included.

In [ ]:
rq6 = rq6_opening_ply(df)
rq6.to_csv(os.path.join(OUTPUT_TABLES, "RQ6_opening_ply.csv"), index=False)
display(rq6)
plot_opening_ply(rq6)

## RQ7: Model Comparison → Predicting Game Outcome

Three classifiers are trained on [white_rating, black_rating, rating_diff, opening_ply] to predict winner (white/black/draw).
Evaluation uses 5-fold stratified CV and a held-out test set.

In [ ]:
X, y, le = prepare_features(df)
print(f"Features: {["white_rating", "black_rating", "rating_diff", "opening_ply"]}")
print(f"Classes: {list(le.classes_)}")
print(f"Class distribution: {dict(zip(le.classes_, np.bincount(y)))}")

In [ ]:
rq7, detailed = train_and_evaluate(X, y, le)
rq7.to_csv(os.path.join(OUTPUT_TABLES, "RQ7_model.csv"), index=False)
display(rq7)
plot_model_comparison(rq7)

### Classification Reports (precision, recall, F1 per class)

In [ ]:
for name, info in detailed.items():
    print(f"
{"="*50}")
    print(f"{name}")
    print("="*50)
    display(info["classification_report"])
    info["classification_report"].to_csv(
        os.path.join(OUTPUT_TABLES, f"RQ7_report_{name.replace(" ", "_")}.csv")
    )

### Confusion Matrices

In [ ]:
plot_confusion_matrices(detailed, list(le.classes_))

### Feature Importances (tree-based models)

In [ ]:
plot_feature_importances(detailed)